# Notebook 02a — SHAP Explanations

This notebook generates SHAP explanations for 500 sampled test instances.

**Inputs** (from Drive):
- `adult_test.csv` — encoded test set with sensitive columns attached
- `model_rf.pkl` — trained Random Forest classifier
- `feature_names.pkl` — list of 14 feature names

**Outputs** (saved to Drive):
- `sample_indices.npy` — row indices of the 500 sampled instances (used by 02b)
- `X_sample.csv` — feature matrix for the 500 instances
- `y_sample.csv` — true labels for the 500 instances
- `sens_sample.csv` — sensitive attributes (sex_raw, race_raw, age_group)
- `shap_values.npy` — SHAP values, shape (500, 14)

## Step 1 — Install and Import Libraries

In [ ]:
!pip install shap -q

In [ ]:
import numpy as np
import pandas as pd
import pickle
import os
import shap
import matplotlib.pyplot as plt
from google.colab import drive

print(f"shap version: {shap.__version__}")

## Step 2 — Mount Drive and Load Artifacts

In [ ]:
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/XAIP/data/'
os.makedirs(BASE, exist_ok=True)
print(f"✓ Drive mounted — base path: {BASE}")

In [ ]:
# Load test set
df_test = pd.read_csv(BASE + 'adult_test.csv')

# Load model
with open(BASE + 'model_rf.pkl', 'rb') as f:
    model = pickle.load(f)

# Load feature names
with open(BASE + 'feature_names.pkl', 'rb') as f:
    feature_names = pickle.load(f)

# Separate features, labels, and sensitive columns
X_test      = df_test[feature_names].copy()
y_test      = df_test['income'].copy()
sens_test   = df_test[['sex_raw', 'race_raw', 'age_group']].copy()

print(f"Test set shape:      {X_test.shape}")
print(f"Feature names ({len(feature_names)}): {feature_names}")
print(f"Target distribution:\n{y_test.value_counts()}")

## Step 3 — Sample 500 Instances

We stratify by income class to preserve the ~75/25 class split.
The resulting indices are saved so notebook 02b uses the exact same instances.

In [ ]:
N_SAMPLE    = 500
RANDOM_SEED = 42

# Stratified sample: draw proportionally from each income class
rng = np.random.default_rng(RANDOM_SEED)

idx_neg = y_test[y_test == 0].index.to_numpy()
idx_pos = y_test[y_test == 1].index.to_numpy()

n_pos = int(round(N_SAMPLE * len(idx_pos) / len(y_test)))
n_neg = N_SAMPLE - n_pos

chosen_neg = rng.choice(idx_neg, size=n_neg, replace=False)
chosen_pos = rng.choice(idx_pos, size=n_pos, replace=False)
sample_idx = np.sort(np.concatenate([chosen_neg, chosen_pos]))

print(f"Total sampled:        {len(sample_idx)}")
print(f"  <=50K (class 0):    {n_neg}")
print(f"  >50K  (class 1):    {n_pos}")

# Save indices for 02b
np.save(BASE + 'sample_indices.npy', sample_idx)
print(f"\n✓ sample_indices.npy saved")

## Step 4 — Build and Save Sample Dataframes

In [ ]:
X_sample    = X_test.loc[sample_idx].reset_index(drop=True)
y_sample    = y_test.loc[sample_idx].reset_index(drop=True)
sens_sample = sens_test.loc[sample_idx].reset_index(drop=True)

# Alignment check
assert len(X_sample) == len(y_sample) == len(sens_sample) == N_SAMPLE

print(f"X_sample shape:   {X_sample.shape}")
print(f"y_sample dist:\n{y_sample.value_counts()}")
print(f"\nSensitive attributes:")
print(f"  sex_raw:   {sens_sample['sex_raw'].value_counts().to_dict()}")
print(f"  race_raw:  {sens_sample['race_raw'].value_counts().to_dict()}")
print(f"  age_group: {sens_sample['age_group'].value_counts().sort_index().to_dict()}")

X_sample.to_csv(BASE + 'X_sample.csv', index=False)
y_sample.to_csv(BASE + 'y_sample.csv', index=False)
sens_sample.to_csv(BASE + 'sens_sample.csv', index=False)

print("\n✓ X_sample.csv, y_sample.csv, sens_sample.csv saved")

## Step 5 — Compute SHAP Values

We use `TreeExplainer`, which computes exact Shapley values for tree-based models.
For binary classification, SHAP returns values for both classes;
we take `shap_values[:, :, 1]` (positive class: >50K) → shape (500, 14).

`check_additivity=False` avoids floating-point precision errors that
sometimes occur with sklearn's RandomForest.

In [ ]:
explainer = shap.TreeExplainer(model)
print("✓ TreeExplainer initialized")
print(f"  Expected value (base rate): {explainer.expected_value}")

In [ ]:
shap_output = explainer.shap_values(X_sample, check_additivity=False)

# shap_output is a list [values_class0, values_class1] for binary classifiers
# or a 3-D array of shape (500, 14, 2) depending on shap version
if isinstance(shap_output, list):
    shap_vals = shap_output[1]          # positive class
else:
    shap_vals = shap_output[:, :, 1]    # positive class

print(f"SHAP values shape: {shap_vals.shape}")
assert shap_vals.shape == (N_SAMPLE, len(feature_names)), \
    f"Unexpected shape: {shap_vals.shape}"
print("✓ Shape check passed")

## Step 6 — Save SHAP Values

In [ ]:
np.save(BASE + 'shap_values.npy', shap_vals)
print(f"✓ shap_values.npy saved — shape {shap_vals.shape}, dtype {shap_vals.dtype}")

# Sanity check: reload and verify
shap_reload = np.load(BASE + 'shap_values.npy')
assert np.allclose(shap_vals, shap_reload)
print("✓ Reload verification passed")

## Step 7 — Visualizations

### 7a — Summary Plot (Beeswarm)
Shows feature impact direction and magnitude across all 500 instances.

In [ ]:
shap.summary_plot(
    shap_vals,
    X_sample,
    feature_names=feature_names,
    show=True
)

### 7b — Global Feature Importance (Mean |SHAP|)

In [ ]:
mean_abs_shap = np.abs(shap_vals).mean(axis=0)
importance_df = pd.DataFrame({
    'feature': feature_names,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(importance_df['feature'], importance_df['mean_abs_shap'], color='steelblue')
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('SHAP Global Feature Importance (n=500)')
plt.tight_layout()
plt.savefig(BASE + 'shap_importance.png', dpi=150)
plt.show()

print("\nTop 5 features by mean |SHAP|:")
print(importance_df.sort_values('mean_abs_shap', ascending=False).head(5).to_string(index=False))

### 7c — Example: SHAP Waterfall for a Single Instance

In [ ]:
# Pick one high-confidence positive prediction to show as an example
pred_proba = model.predict_proba(X_sample)[:, 1]
example_idx = int(np.argmax(pred_proba))

print(f"Example instance index: {example_idx}")
print(f"  Predicted P(>50K):  {pred_proba[example_idx]:.3f}")
print(f"  True label:         {y_sample.iloc[example_idx]}")

# Build Explanation object for waterfall plot
base_val = explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) \
           else explainer.expected_value

explanation = shap.Explanation(
    values=shap_vals[example_idx],
    base_values=base_val,
    data=X_sample.iloc[example_idx].values,
    feature_names=feature_names
)
shap.waterfall_plot(explanation)

## Step 8 — Summary

In [ ]:
print("=" * 60)
print("  SUMMARY — OUTPUTS FOR NOTEBOOK 02b AND 03")
print("=" * 60)
print(f"\nBASE PATH:\n  {BASE}")
print(f"\nFILES PRODUCED:")
files = ['sample_indices.npy', 'X_sample.csv', 'y_sample.csv',
         'sens_sample.csv', 'shap_values.npy', 'shap_importance.png']
for fname in files:
    fpath = BASE + fname
    if os.path.exists(fpath):
        size_kb = os.path.getsize(fpath) / 1024
        print(f"  {fname:30s}  {size_kb:.1f} KB")
print(f"\nSHAP VALUES:")
print(f"  Shape:  {shap_vals.shape}  (instances × features)")
print(f"  dtype:  {shap_vals.dtype}")
print(f"  min:    {shap_vals.min():.4f}")
print(f"  max:    {shap_vals.max():.4f}")
print(f"  mean|v|:{np.abs(shap_vals).mean():.4f}")
print(f"\n✓ SHAP complete — ready for 02b (LIME) and 03 (metrics)")